# Signal & Background Distributions -- DATA (After Cosmic Tagger)

The DATA counterpart of `SignalBackground_Distributions_AfterCosmic.ipynb`, and the
cosmic-tagger-applied sibling of `SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb`
-- same code as that notebook, with the COSMIC TAGGER CUT turned on. That
notebook's header explained why the tagger was left out: "the tagger flags are
real-data quantities too and could be applied to data in principle, but that
is a decision for a separate run, not a default this notebook makes
silently." This is that separate run.

Real detector data (the off-beam / beam-off sample) still has no simulated
interaction to record: no sed-smear/sed-sce truth, no mc.json, no interaction
channel, no vertex. Every plot and every cut that depends on a true cluster is
therefore GONE, not skipped, exactly as in the Before-tagger notebook -- see
its header for the full kept/gone breakdown. The only thing that changes here
is one more reco-level cut.

**What survives, reco-only:**

| kept | why |
|---|---|
| reading `img-global`, `clustering-global`, `op` (`read_data_files_for_event`, readfiles.py) | pure reco/optical, no truth file involved |
| beam-window cut (`AfterBeamWindowCut`) | bridged optical flash time, not a truth quantity |
| **the cosmic tagger cut** | the WireCell taggers' flags are written against RECO clusters and real_cluster_id, both reco-side quantities -- `read_data_files_for_event` already returns `taggers` for real data (see its docstring), nothing about applying `apply_cosmic_tagger_cut` requires truth |
| `RECO_ID_FIELD` grouping choice | how clustering-global's points become reco clusters, a reco-side decision |
| reco cluster selection flow (`DrawRecoClusterSelectionFlow`) | already truth-free upstream: "no cosmic/neutrino split -- reco clusters carry no truth label" |

**What is still gone, and why:** unchanged from the Before-tagger notebook --
true-cluster stack, completeness/purity/pairing, `categorize_reco_clusters`,
`selection_efficiency/`, `signal_neutrino_multiplicity/`,
`energy_reconstruction/`, `Saved_Clusters/`, `selection_completeness_vs_purity/`
-- all need a true cluster to compare against or a true denominator, and real
data has neither.

**The cosmic tagger cut itself**

Applied to the `AfterBeamWindowCut` selection only, via `apply_cosmic_tagger_cut`
(selections.py) -- byte-for-byte the same call the truth-bearing notebook
makes, just without a true neutrino on the other side of it to report losing.
PER-FLASH (`selections.COSMIC_TAG_PROPAGATE_SCOPE = 'flash'`): the tagged
cluster and anything sharing its flash are removed; activity at a different
flash time in the same event survives. `Apply_cosmic_tagger_cut` in the
configuration cell is a toggle (default `True`) so a run comparing with/without
the cut stays one flag away, the same convention every other cosmic-tagger
notebook in this codebase uses.

**The one plot this notebook draws**

`draw_selected_reco_energy` (`AnalysisDistributions/draw_selection_performance.py`):
every reco cluster that survives the beam-window cut AND the cosmic tagger cut,
binned in reco cluster energy from its charge
(`RECO_WORK_FUNCTION_EV * charge / RECO_RECOMBINATION_FACTOR`), drawn as DATA
POINTS -- a single BLACK marker at each bin centre, height = number of reco
clusters in that bin, with a vertical statistical (`sqrt(N)`) uncertainty bar.
Empty bins are left undrawn rather than plotted at zero. No fill, no
connecting line, no histogram bars: a measured count is not a modelled
composition, so it is not drawn as one.

Drawn at both entries of `BIN_WIDTHS_MEV` (200 and 100 MeV), each in its own
`selection_reco/<width>MeV/` directory. Each width directory also gets a
per-bin count table (`selected_reco_info_*.txt`) and a ROOT histogram
(`selected_reco_histograms_*.root`, written with `uproot`, NOT PyROOT -- see
`write_signal_background_root`'s docstring for why PyROOT is unavailable on
this machine) so the figures can be restyled later without re-running the job.

**Input sample and run scope**

The off-beam (beam-off) sample: `r3-beam-off-2026-09-11/bee`,
one real-data event per zip (`bee_r<run>_s<subrun>_e<event>.zip`), grouped into
`chunk_00..chunk_09`, each with `subchunk_00..subchunk_09` of 10 zips (see
`chunk_offbeam_sample.py`). `stage_nuecc_chunks()` (readfiles.py) is reused
unchanged -- it is a generic one-event-per-zip stager despite the name -- to lay
a chunk's subchunks out as `<chunk>__<subchunk>/data/<k>/` trees.

**This run is the FULL SAMPLE** (`DATA_SOURCE_CHUNKS`, `DATA_N_FILES` below):
all 10 source chunks (`chunk_00..chunk_09`), each staged whole (100 events/chunk),
1000 events total. The first test ran `chunk_00/subchunk_00` only (10 events)
and was confirmed before widening to this -- same test-then-widen path
`SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb` and its
on-beam sibling both took.

**Output** -- JOB LEVEL ONLY, written to
`AnalysisDistributions/multi_file_plots_charge_light_matching/Signal_Background_Distributions_AfterCosmicTagger_Data_OffBeam/combined_apa_<date>_<time>/job_summary/`:

```
selection_reco/
  100MeV/
    selected_reco_energy_AfterBeamWindowCut_100MeV_job_Combined.png
    selected_reco_info_AfterBeamWindowCut_100MeV.txt
    selected_reco_histograms_100MeV.root
  200MeV/
    selected_reco_energy_AfterBeamWindowCut_200MeV_job_Combined.png
    selected_reco_info_AfterBeamWindowCut_200MeV.txt
    selected_reco_histograms_200MeV.root
  reco_cluster_selection_flow_job_Combined.png
summary.txt
```

In [ ]:
# Run scope -- same knobs as the truth-bearing notebook, kept for the same
# reason: selective filtering is useful on any sample, truth or not.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
target_file  = None   # Set to "chunk_00__subchunk_00", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

target_event_range = None   # e.g. (1, 5) for events 1..5
target_file_range = None    # e.g. (0, 4) for the first 5 staged units

if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")

# ========================================================================
# NO LEVEL SWITCHES -- this notebook draws the JOB-LEVEL histogram only.
# ========================================================================
# Same reasoning as the truth-bearing notebook: a per-event or per-file
# histogram of one or two reco clusters is not a composition worth drawing.
# The file/event loops below still run -- they are how the clusters are
# collected -- they just do not draw or write anything of their own.

In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions/, one level below the
# repository root where the pipeline modules and the input trees are. Resolve
# both explicitly so the notebook runs whether Jupyter was started in this
# directory (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions":
    NB_DIR = NB_DIR / "AnalysisDistributions"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")

In [ ]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. Only the
# RECO/OPTICAL side of the chain is needed here: real data has no truth file
# for anything on the true side (build_true_points_charge_light,
# cluster_category, completeness/purity, 1-to-1 pairing, mc.json vertices) to
# read, so none of that is imported. apply_cosmic_tagger_cut IS imported here,
# unlike the Before-tagger notebook -- it is a reco-side cut (tagger flags
# matched to reco clusters by real_cluster_id) and needs no truth.
from readfiles import stage_nuecc_chunks, read_data_files_for_event
from selections import GroupClustersByID, apply_cosmic_tagger_cut
from metadata import build_cluster_flash_metadata, build_img_cluster_flash_metadata
from DrawRecoTrueClusterCount import DrawRecoClusterSelectionFlow
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# Per-cluster record builder shared with every other reco-space notebook, so a
# cluster's total_charge / reco energy means exactly the same thing everywhere.
from draw_variables import build_reco_cluster_variable_records

# Reco-energy conversion and the two bin widths every selection_reco/ figure
# is drawn at -- shared with the truth-bearing notebook's module so a change
# to either stays in sync with both.
from draw_signal_background import (
    BIN_WIDTHS_MEV, DEFAULT_RECO_CUTS_LABEL, reco_cluster_energy_mev,
    RECO_WORK_FUNCTION_EV, RECO_RECOMBINATION_FACTOR,
)

# Selection performance: reco-space output layout, the reco cluster selection
# flow, and the no-truth plot/info/root writers this notebook actually uses.
# categorize_reco_clusters, draw_reco_selection_stack, build_selection_efficiency
# and everything completeness/purity are NOT imported -- they all require a
# true cluster to categorise or score a reco cluster against, which real data
# does not have.
from draw_selection_performance import (
    plot_directory, draw_selected_reco_energy, write_selected_reco_info,
    write_selected_reco_root,
)

In [ ]:
# Configuration: Parent directory containing multiple staged unit subdirectories
# (chunk_00__subchunk_00/, chunk_00__subchunk_01/, ...)
#
# Expected structure (per staged unit). stage_nuecc_chunks() below writes it.
# PARENT_DIR/
#   chunk_00__subchunk_00/data/0/0-img-global.json         (reco clusters)
#   chunk_00__subchunk_00/data/0/0-clustering-global.json  (post charge-light-matching reco clusters)
#   chunk_00__subchunk_00/data/0/0-op.json                 (optical flashes)
#   chunk_00__subchunk_00/data/1/, 2/, ... (one subdirectory per event)
#
# NO sed-smear/sed-sce truth file and NO mc.json exist anywhere in this tree --
# this is real data, not a simulated production -- so read_data_files_for_event
# (readfiles.py) does not look for them.

# ========================================================================
# INPUT SAMPLE -- the off-beam (beam-off) real-data sample: ONE event per zip
# (bee_r<run>_s<subrun>_e<event>.zip, contents at data/0/0-*.json; the "0" is
# NOT the event number, the real (run, subrun, event) is in the zip name).
# stage_nuecc_chunks() rewrites the first DATA_N_FILES zips of each source
# chunk, in groups of DATA_CHUNK_SIZE, into <chunk>__<subchunk>/data/<k>/ under
# DATA_STAGING_ROOT -- the SAME staging function the nuecc/numucc samples use
# (it is a generic one-event-per-zip stager despite the name), so this loop is
# byte-for-byte the same shape as theirs.
# ========================================================================
SAMPLE_NAME = "OffBeam_Sample"
DATA_BEE_ROOT = Path("/Volumes/My Passport/Research_Life/Experiment/SBND/"
                     "Wirecell_Reconstruction/Samples/"
                     "r3-beam-off-2026-09-11/bee")

# FULL SAMPLE -- all 10 source chunks (chunk_00..chunk_09), each staged whole
# (all 10 subchunks x 10 files = 100 events/chunk), 1000 events total. This
# was chunk_00/subchunk_00 only (10 events) for the first test; widened here
# per request now that the test output was confirmed.
DATA_SOURCE_CHUNKS = [f"chunk_{i:02d}" for i in range(10)]   # bee/chunk_NN dir(s) this job runs
DATA_STAGING_ROOT  = DATA_BEE_ROOT.parent / "staging"
DATA_N_FILES       = 100              # per source chunk (10 = subchunk_00 only; 100 = whole chunk)
DATA_CHUNK_SIZE    = 10               # events per staged subchunk

PARENT_DIR = DATA_STAGING_ROOT

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory -- the exact path requested, alongside the other
# multi_file_plots_charge_light_matching outputs.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / "Signal_Background_Distributions_AfterCosmicTagger_Data_OffBeam" / "NuMuCC_Sample"
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\nSELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS -- RECO-LEVEL ONLY. There is no true side to select on,
# so the true-cluster cutoffs, the fiducial vertex bounds and the
# completeness/purity matching radii from the truth-bearing notebook do not
# appear here: they would have nothing to act on.
# ========================================================================

# WHICH ID FIELD defines a reco cluster in clustering-global. Same choice,
# same reasoning, as SignalBackground_Distributions_BeforeCosmicTagger.ipynb:
#   'cluster_id'      -> the COARSE grouping (everything one flash ties together)
#   'real_cluster_id' -> the FINER grouping used before 2026-08-15
RECO_ID_FIELD = 'cluster_id'

# ========================================================================
# RECO SELECTIONS -- reco-level cuts only (there is no true side to select
# on), PLUS the cosmic tagger cut. This is the one difference from
# SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam.ipynb: that
# notebook stopped at the beam-window cut by request; this one goes one cut
# further.
# ========================================================================
RECO_SELECTION_NOCUTS = DEFAULT_RECO_CUTS_LABEL       # 'NoCuts': every reco cluster
RECO_SELECTION_BEAM   = 'AfterBeamWindowCut'          # flash time inside the beam window
RECO_SELECTION_LABELS = [RECO_SELECTION_NOCUTS, RECO_SELECTION_BEAM]

# COSMIC TAGGER CUT: drop in-beam activity the WireCell cosmic taggers
# flagged. Applied to the AfterBeamWindowCut selection only -- same cut, same
# reasoning, as SignalBackground_Distributions_AfterCosmic.ipynb's Apply_cosmic_tagger_cut,
# just with no true neutrino on the other side of it to report losing.
#
# PER-FLASH (selections.COSMIC_TAG_PROPAGATE_SCOPE = 'flash'): the tagged
# cluster and anything sharing its flash are removed; activity at a different
# flash time in the same event is KEPT.
Apply_cosmic_tagger_cut = True

print("\nCuts applied: RECO-LEVEL ONLY (no truth exists to cut on)")
print(f"- reco cluster id field: {RECO_ID_FIELD}")
print(f"- beam window: {BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us (bridged flash time)")
print(f"- cosmic tagger cut: {'applied' if Apply_cosmic_tagger_cut else 'NOT applied'} (per-flash)")
print(f"- reco energy = {RECO_WORK_FUNCTION_EV} eV * charge / {RECO_RECOMBINATION_FACTOR}")
print(f"\nReco selections counted: " + ", ".join(RECO_SELECTION_LABELS))
print(f"'Selected' (the plotted population) means: {RECO_SELECTION_BEAM}"
     f"{' + cosmic tagger cut' if Apply_cosmic_tagger_cut else ''}")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Idempotent -- a data/<k>/ that already holds files is left alone -- so
# re-running this notebook never re-stages an already-staged unit.
staging_start = time.time()
staged_chunks = []
for source_chunk in DATA_SOURCE_CHUNKS:
    print(f"Staging {DATA_N_FILES} zips from {DATA_BEE_ROOT / source_chunk} "
          f"in subchunks of {DATA_CHUNK_SIZE}")
    staged_chunks += stage_nuecc_chunks(
        DATA_BEE_ROOT / source_chunk, DATA_STAGING_ROOT,
        n_files=DATA_N_FILES, chunk_size=DATA_CHUNK_SIZE)
print(f"Staged {len(staged_chunks)} subchunk dir(s) from "
      f"{len(DATA_SOURCE_CHUNKS)} source chunk(s)")
staging_seconds = time.time() - staging_start

In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories.
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name, or None if it has no trailing
    digits. Used by target_file_range so files are selected by their real
    index rather than by position in the lexicographically sorted list.
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# staging/ can still hold units from an earlier run with a larger DATA_N_FILES
# -- keep only the ones stage_nuecc_chunks() just wrote.
staged_names = {d.name for d in staged_chunks}
input_directories = [d for d in input_directories if d.name in staged_names]
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")

In [ ]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA selected-reco-energy histogram, DATA,
# AFTER the cosmic tagger cut
# ============================================================================
# Reco-only. read_data_files_for_event returns 'reco' (img-global), 'op',
# 'clustering' (clustering-global) and 'taggers' -- there is no
# 'true'/'true_clustering'/'mc' key to unpack, because none of those files
# exist for real data. 'taggers' is what the cosmic tagger cut below reads.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

# Reco cluster selection flow: how many reco clusters survive each stage,
# counted in whichever namespace RECO_ID_FIELD selects.
RECO_FLOW_STAGES = [(RECO_SELECTION_NOCUTS, 'No cuts'),
                    (RECO_SELECTION_BEAM,   '+ Beam window')]
job_reco_flow_counts = {label: 0 for label, _ in RECO_FLOW_STAGES}
job_reco_var_records = {label: [] for label in RECO_SELECTION_LABELS}
# What the cosmic tagger cut removed, for summary.txt -- same counters,
# same reasoning, as SignalBackground_Distributions_AfterCosmic.ipynb: it is a large cut
# and a job that silently dropped a chunk of its in-beam clusters should say so.
n_events_cosmic_tagged   = 0
n_clusters_cosmic_tagged = 0

total_events_processed = 0
total_files_processed  = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    for evt in range(event_low, event_high):
        if target_event is not None and evt != target_event:
            continue
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_data_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"
        op_data = result['op']

        # ------------------------------------------------------------------
        # RECO CLUSTERS. clustering-global (post charge-light matching)
        # grouped by RECO_ID_FIELD, and nothing else: no beam-window cut, no
        # fiducial cut, no minimum point count. That is what
        # RECO_CUTS_LABEL = 'NoCuts' names.
        # ------------------------------------------------------------------
        x_clu, y_clu, z_clu, id_clu, q_clu, real_id_clu = result['clustering']
        reco_ids_clu = id_clu if RECO_ID_FIELD == 'cluster_id' else real_id_clu
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, reco_ids_clu, q_clu))

        # BEAM-WINDOW CUT, the second (and only other) selection. op.json
        # flashes are attached to img-global clusters and then bridged onto
        # clustering-global clusters by point charge; a reco cluster passes if
        # its bridged flash time falls in [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US].
        # A cluster with NO bridged flash is CUT: it has no time, so it cannot
        # be shown to be in the beam window.
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)
        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        real_to_coarse = {float(r): float(c) for r, c in zip(real_id_clu, id_clu)}
        if RECO_ID_FIELD == 'cluster_id':
            clu_beam_window_ids = {real_to_coarse[r] for r in clu_beam_window_ids
                                   if r in real_to_coarse}

        if len(predicted_points) and clu_beam_window_ids:
            beam_ids_array = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points_beam = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
        else:
            predicted_points_beam = predicted_points[:0]

        # COSMIC TAGGER CUT, on the beam-window survivors and nothing else --
        # byte-for-byte the same call SignalBackground_Distributions_AfterCosmic.ipynb
        # makes, just with no true neutrino on the other side of it to report
        # losing.
        #
        # PER-FLASH: the tagged cluster and anything sharing its flash are
        # removed; activity at other flash times is kept. Set
        # selections.COSMIC_TAG_PROPAGATE_SCOPE='event' for all-or-nothing.
        if Apply_cosmic_tagger_cut:
            predicted_points_beam, tagger_cut_info = apply_cosmic_tagger_cut(
                predicted_points_beam, result.get('taggers'))
            if tagger_cut_info['n_tagged_clusters']:
                n_events_cosmic_tagged += 1
                n_clusters_cosmic_tagged += tagger_cut_info['n_tagged_clusters']
                print(f"    cosmic tagger cut: removed "
                      f"{tagger_cut_info['n_tagged_clusters']} in-beam cluster(s) "
                      f"({tagger_cut_info['n_direct']} tagged directly by "
                      f"{', '.join(tagger_cut_info['tagged_by'])}), "
                      f"{tagger_cut_info['n_points_removed']} points")

        event_reco_points_by_selection = {
            RECO_SELECTION_NOCUTS: predicted_points,
            RECO_SELECTION_BEAM:   predicted_points_beam,
        }
        event_reco_var_records = {}
        for selection_label, selection_points in event_reco_points_by_selection.items():
            selection_clusters = GroupClustersByID(selection_points) if len(selection_points) else {}
            event_reco_var_records[selection_label] = build_reco_cluster_variable_records(
                selection_clusters, input_file_name, evt, event_key, "Combined")

        # ------------------------------------------------------------------
        # AGGREGATE TO JOB LEVEL
        # ------------------------------------------------------------------
        for selection_label in RECO_SELECTION_LABELS:
            job_reco_var_records[selection_label].extend(event_reco_var_records[selection_label])
        for selection_label, _stage in RECO_FLOW_STAGES:
            job_reco_flow_counts[selection_label] += len(event_reco_var_records[selection_label])

        print(
            f"  Event {evt}: "
            f"reco clusters={len(event_reco_var_records[RECO_SELECTION_NOCUTS])} "
            f"(in beam window={len(event_reco_var_records[RECO_SELECTION_BEAM])})"
        )
        total_events_processed += 1

    # Per-unit status, so a long multi-chunk job shows a milestone as each
    # staged chunk__subchunk finishes.
    _src_chunk = input_file_name.split("__")[0]
    print(f"  ==> unit {file_idx + 1}/{len(input_directories)} done: {input_file_name}  "
          f"({total_events_processed} events, "
          f"{len(job_reco_var_records[RECO_SELECTION_BEAM])} selected reco clusters, "
          f"{(time.time() - job_start_time) / 60:.1f} min elapsed)")
    _next = input_directories[file_idx + 1].name if file_idx + 1 < len(input_directories) else ""
    if _src_chunk and not _next.startswith(_src_chunk + "__"):
        print(f"  ===================  SOURCE CHUNK {_src_chunk} COMPLETE  "
              f"({total_events_processed} events so far)  ===================")

# ============================================================================
# JOB-LEVEL: every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total selected reco clusters ({RECO_SELECTION_BEAM}): {len(job_reco_var_records[RECO_SELECTION_BEAM])}")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# THE ONE PLOT -- selected reco clusters vs reco cluster energy, at each bin
# width, plus its ROOT histogram and per-bin count table. Drawn as black data
# points at bin centres with statistical (sqrt(N)) error bars -- see
# draw_selected_reco_energy's docstring for why (no fill, no bars: a measured
# count, not a modelled composition).
# ============================================================================
root_paths = {}
for bin_width in BIN_WIDTHS_MEV:
    reco_dir = plot_directory(job_output_dir, 'selection_reco', bin_width)

    all_energies = draw_selected_reco_energy(
        job_reco_var_records[RECO_SELECTION_BEAM], reco_dir, "Job Level", "job", "Combined",
        bin_width=bin_width, reco_cuts_label=RECO_SELECTION_BEAM)

    write_selected_reco_info(all_energies, reco_dir, "Job Level",
                             bin_width=bin_width, reco_cuts_label=RECO_SELECTION_BEAM)
    root_paths[bin_width] = write_selected_reco_root(
        all_energies, reco_dir, bin_width=bin_width, reco_cuts_label=RECO_SELECTION_BEAM)
    print(f"  {bin_width:.0f} MeV: {len(all_energies)} selected reco clusters -> {root_paths[bin_width]}")

# Reco cluster selection flow, one Total bar per stage -- already truth-free
# upstream (DrawRecoTrueClusterCount.py: "no cosmic/neutrino split -- reco
# clusters carry no truth label").
DrawRecoClusterSelectionFlow(
    [{'key': label, 'stage': stage, 'geometry': False,
      'total': job_reco_flow_counts[label]} for label, stage in RECO_FLOW_STAGES],
    plot_directory(job_output_dir, 'selection_reco'), "Job Level", "job", "Combined",
    include_geometry_cuts=False)
print(f"  reco selection flow: "
      f"{', '.join(f'{s}={job_reco_flow_counts[l]}' for l, s in RECO_FLOW_STAGES)}")

# ============================================================================
# JOB SUMMARY TEXT FILE
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- SELECTED RECO ENERGY, DATA (no truth available), AFTER COSMIC TAGGER")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
summary_lines.append("")
summary_lines.append("Input definitions:")
summary_lines.append(f"  reco cluster id field:    {RECO_ID_FIELD}"
                     f"   (clustering-global; "
                     f"{'coarse -- all activity on one flash is ONE cluster' if RECO_ID_FIELD == 'cluster_id' else 'fine -- flash-mates stay separate'})")
summary_lines.append(f"  reco cluster file:        clustering-global")
summary_lines.append(f"  true cluster file:        NONE -- real data has no truth")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (RECO-LEVEL ONLY -- there is no true side to cut on):")
summary_lines.append(f"  RECO selections (counted): {', '.join(RECO_SELECTION_LABELS)}")
summary_lines.append(f"    {RECO_SELECTION_NOCUTS}: every reco cluster; no beam-window, "
                     f"fiducial or point-count cut")
summary_lines.append(f"    {RECO_SELECTION_BEAM}: bridged flash time in "
                     f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us; a cluster with no flash is cut")
summary_lines.append(f"  cosmic tagger cut:        "
                     f"{'applied, per-flash (selections.COSMIC_TAG_PROPAGATE_SCOPE)' if Apply_cosmic_tagger_cut else 'NOT applied'}")
summary_lines.append(f"  reco energy estimate:     {RECO_WORK_FUNCTION_EV} eV * charge / "
                     f"{RECO_RECOMBINATION_FACTOR} = {reco_cluster_energy_mev(1.0):.6g} MeV per unit charge")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
for selection_label in RECO_SELECTION_LABELS:
    summary_lines.append(f"Total reco clusters, {selection_label}: "
                         f"{len(job_reco_var_records[selection_label])}")
if Apply_cosmic_tagger_cut:
    summary_lines.append(f"Cosmic tagger cut: removed {n_clusters_cosmic_tagged} in-beam "
                         f"cluster(s) over {n_events_cosmic_tagged} event(s) -- per-flash")
    summary_lines.append("  (COSMIC_TAG_PROPAGATE_SCOPE = 'flash'), so activity at a DIFFERENT")
    summary_lines.append("  flash time is KEPT (see selections.apply_cosmic_tagger_cut)")
else:
    summary_lines.append("Cosmic tagger cut: NOT applied")
summary_lines.append("")
summary_lines.append("The plotted population ('selected') is AfterBeamWindowCut"
                     + (" + cosmic tagger cut" if Apply_cosmic_tagger_cut else "") + ". Its histogram")
summary_lines.append("(both bin widths), ROOT file and per-bin table are under selection_reco/:")
for bin_width in BIN_WIDTHS_MEV:
    summary_lines.append(f"  {bin_width:.0f} MeV -> {root_paths[bin_width]}")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append(f"  of which one-time input staging: {timedelta(seconds=int(staging_seconds))} ({staging_seconds:.1f} s)   -- ~0 on a re-run")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")